# Encoder-FFN Model with Honeypot Deception
## DNS Cache Poisoning Detection in Software-Defined Networks

**Reference paper:** Kengne Tchendji et al. (2026)

**Pipeline:**
1. Environment setup (clone repo + install deps)
2. Imports & configuration
3. Kaggle credentials
4. Download datasets (Marques + BCCC) with cache
5. EDA
6. Preprocessing
7. Train/Val/Test split + scaling + tokenisation
8. Full model training (Marques)
9. Evaluation (Marques)
10. **Ablation study** (Marques) — FFN-Only vs Encoder-Only vs Full
11. Full model training (BCCC)
12. Evaluation (BCCC)
13. **Ablation study** (BCCC)
14. Per-category breakdown (BCCC)
15. **Statistical tests** (McNemar) between branches
16. Final summary
17. Package plots as ZIP for download

## 1. Environment setup — clone repository + install dependencies

On Colab, opening this notebook from GitHub loads **only the notebook**, not the
project package. This cell clones the repository so `from src import ...` works.

In [ ]:
import os, sys, subprocess
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

IS_COLAB = in_colab()
print('Running on Colab:', IS_COLAB)

REPO_URL = 'https://github.com/jeffJeffrey/encoder-ffn-dns.git'
REPO_DIR = '/content/encoder-ffn-dns'

if IS_COLAB:
    if not Path(REPO_DIR).exists():
        print('Cloning repository...')
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    os.environ['PROJECT_ROOT'] = REPO_DIR
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'kagglehub', 'tensorflow', 'scikit-learn',
                    'matplotlib', 'seaborn', 'pandas', 'numpy', 'scipy'], check=False)

print('Working dir:', os.getcwd())
sys.path.insert(0, os.getcwd())

## 2. Imports and configuration

In [ ]:
import gc
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

from src import config
from src import data_loader, preprocessing, eda, train, evaluate
from src.models import build_encoder_ffn, build_ffn_only, build_encoder_only
from src.utils import save_json, measure_latency, count_params

config.set_global_seed()
print('TensorFlow :', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))
print('QUICK_TEST :', config.QUICK_TEST)
print('Figures dir:', config.FIGURES_DIR)
print('Results dir:', config.RESULTS_DIR)

## 3. Kaggle credentials

Required to download BCCC-CIC-Bell-DNS-2024. Two options:
- **Colab Secrets**: add `KAGGLE_USERNAME` + `KAGGLE_KEY` in the 🔑 panel.
- **Local**: place `kaggle.json` at `~/.kaggle/kaggle.json` (chmod 600).

In [ ]:
import shutil, json

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

def setup_kaggle():
    tok = _secret('KAGGLE_API_TOKEN')
    if tok:
        os.environ['KAGGLE_API_TOKEN'] = tok
        print('Loaded KAGGLE_API_TOKEN from Colab Secrets.'); return
    u, k = _secret('KAGGLE_USERNAME'), _secret('KAGGLE_KEY')
    if u and k:
        os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = u, k
        print('Loaded KAGGLE_USERNAME / KAGGLE_KEY from Colab Secrets.'); return
    kpath = Path.home() / '.kaggle' / 'kaggle.json'
    if kpath.exists():
        print('Found existing ~/.kaggle/kaggle.json.'); return
    try:
        from google.colab import files
        print('Upload your kaggle.json (Settings > API > Create Legacy API Key):')
        uploaded = files.upload()
        kpath.parent.mkdir(parents=True, exist_ok=True)
        for name in uploaded:
            shutil.move(name, kpath)
        kpath.chmod(0o600)
        creds = json.loads(kpath.read_text())
        os.environ['KAGGLE_USERNAME'] = creds['username']
        os.environ['KAGGLE_KEY'] = creds['key']
        print('kaggle.json installed.')
    except Exception as e:
        print('Could not set up Kaggle credentials:', e)

setup_kaggle()

## 4. Download datasets (with cache)

In [ ]:
# Marques et al. (GitHub, ~90k rows)
df_marques_raw = data_loader.get_marques()

# BCCC-CIC-Bell-DNS-2024 (Kaggle, ~965k rows)
df_bccc_raw = data_loader.get_bccc()

## 5. Exploratory Data Analysis

In [ ]:
eda.eda_marques(df_marques_raw,
                figures_dir=config.FIGURES_DIR / 'eda' / 'marques')

eda.eda_bccc(df_bccc_raw,
             figures_dir=config.FIGURES_DIR / 'eda' / 'bccc')

## 6. Preprocessing

In [ ]:
# Marques
X_num_m, X_text_m, y_m, num_cols_m, text_cols_m = \
    preprocessing.preprocess_marques(df_marques_raw)

# BCCC (also returns per-category labels for ablation and eval)
X_num_c, X_text_c, y_c, num_cols_c, text_cols_c, bccc_categories = \
    preprocessing.preprocess_bccc(df_bccc_raw)

# Free raw frames
del df_marques_raw, df_bccc_raw
gc.collect()

## 7. Train / Val / Test split + scaling + tokenisation

**Split note:** We use an 80/10/10 (train/val/test) stratified split.
The paper describes an 80/20 split for offline training; here the 10% validation
set is used for early-stopping and the 10% test set is the *held-out* set used
for all reported metrics. This explains why test-set sizes in Tables 2–3 are
smaller than 20% of the total dataset.

In [ ]:
data_m = preprocessing.full_pipeline(
    X_num_m, X_text_m, y_m,
    dataset_name='Marques',
    max_vocab=config.MAX_VOCAB_MARQUES
)

data_c = preprocessing.full_pipeline(
    X_num_c, X_text_c, y_c,
    dataset_name='BCCC',
    max_vocab=config.MAX_VOCAB_BCCC
)

print(f'\nMarques — n_num: {data_m["n_num_features"]}, vocab: {data_m["vocab_size"]}')
print(f'BCCC    — n_num: {data_c["n_num_features"]}, vocab: {data_c["vocab_size"]}')

## 8. Full model training — Marques et al.

In [ ]:
print('=' * 60)
print('TRAINING — MARQUES et al.')
print('=' * 60)

model_m = build_encoder_ffn(
    n_num_features=data_m['n_num_features'],
    vocab_size=data_m['vocab_size'],
    name='Encoder_FFN_Marques'
)
model_m.summary()
print(f'Parameters: {count_params(model_m)}')

history_m = train.train_model(model_m, data_m, name='marques')

## 9. Evaluation — Marques et al.

In [ ]:
evaluate.plot_learning_curves(
    history_m, name='Marques',
    figures_dir=config.FIGURES_DIR / 'training' / 'marques'
)

metrics_m = evaluate.evaluate_model(
    model_m, data_m,
    name='Marques',
    dataset_tag='marques',
    figures_dir=config.FIGURES_DIR / 'evaluation' / 'marques'
)
print('Marques metrics:', metrics_m)

# Inference latency
lat_m = measure_latency(model_m,
    input_shape=(config.MAX_SEQ_LEN, data_m['n_num_features']))
print(f'Inference latency: {lat_m["latency_ms"]:.4f} ms/packet')

## 10. Ablation study — Marques et al.

We train three variants to empirically validate that fusing both branches
outperforms either branch alone (Reviewer 1 Comment 7 / Reviewer 2 Comment 2):

| Variant | Branches used | Justification |
|---------|--------------|---------------|
| FFN-Only | Numerical only | Baseline: hand-crafted features |
| Encoder-Only | Textual only | Does text alone suffice? |
| Full (paper) | Both fused | Proposed model |

All variants use the same head architecture and training settings for fair comparison.

In [ ]:
ablation_variants_m = {
    'FFN-Only': build_ffn_only(
        n_num_features=data_m['n_num_features'], name='FFN_Only_Marques'),
    'Encoder-Only': build_encoder_only(
        n_num_features=data_m['n_num_features'],
        vocab_size=data_m['vocab_size'], name='Encoder_Only_Marques'),
    'Full (paper)': model_m,   # already trained
}

ablation_results_m = {}
for vname, vmodel in ablation_variants_m.items():
    if vname == 'Full (paper)':
        # Already trained — just evaluate
        metrics_v = metrics_m
    else:
        print(f'\n--- Ablation variant: {vname} ---')
        train.train_model(vmodel, data_m,
                          name=f'ablation_marques_{vname.lower().replace(" ", "_")}')
        metrics_v = evaluate.evaluate_model(
            vmodel, data_m,
            name=f'Marques — {vname}',
            dataset_tag=f'marques_ablation_{vname.lower().replace(" ","_")}',
            figures_dir=config.FIGURES_DIR / 'ablation' / 'marques'
        )
    ablation_results_m[vname] = metrics_v

ablation_df_m = evaluate.ablation_comparison(
    ablation_results_m, 'Marques',
    figures_dir=config.FIGURES_DIR / 'ablation' / 'marques'
)
print(ablation_df_m)

### McNemar test — Marques ablation

We test whether the performance difference between the Full model and each
single-branch variant is statistically significant (α = 0.05).

In [ ]:
# Obtain test predictions for all three Marques variants
y_true_m = data_m['y_te']

def _preds(model, data, threshold=0.5):
    prob = model.predict([data['X_te_seq'], data['X_te_num']], verbose=0).flatten()
    return (prob >= threshold).astype(int)

preds_m = {vn: _preds(vm, data_m) for vn, vm in ablation_variants_m.items()}

mcnemar_m = {}
for vname in ['FFN-Only', 'Encoder-Only']:
    result = evaluate.mcnemar_test(
        y_true_m,
        preds_m['Full (paper)'],
        preds_m[vname],
        name_a='Full (paper)',
        name_b=vname
    )
    mcnemar_m[f'Full vs {vname}'] = result

save_json(mcnemar_m, config.RESULTS_DIR / 'mcnemar_marques.json')
print('McNemar tests saved.')

## 11. Full model training — BCCC-CIC-Bell-DNS-2024

In [ ]:
print('=' * 60)
print('TRAINING — BCCC-CIC-Bell-DNS-2024')
print('=' * 60)

model_c = build_encoder_ffn(
    n_num_features=data_c['n_num_features'],
    vocab_size=data_c['vocab_size'],
    dropout_rate=config.BCCC_DROPOUT_RATE,
    dropout_mid=config.BCCC_DROPOUT_MID,
    l2_reg=config.BCCC_L2_REG,
    lr=config.BCCC_LR,
    name='Encoder_FFN_BCCC'
)
# Gradient clipping for BCCC stability
model_c.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.BCCC_LR, clipnorm=1.0),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=config.LABEL_SMOOTHING),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)
model_c.summary()
print(f'Parameters: {count_params(model_c)}')

# Class weights for BCCC imbalance
_classes = np.unique(data_c['y_tr'])
_cw = compute_class_weight('balanced', classes=_classes, y=data_c['y_tr'])
class_weight_bccc = {int(c): float(w) for c, w in zip(_classes, _cw)}
print(f'Class weights (balanced): {class_weight_bccc}')

history_c = train.train_model(model_c, data_c, name='bccc',
                               class_weight=class_weight_bccc)

## 12. Evaluation — BCCC

In [ ]:
evaluate.plot_learning_curves(
    history_c, name='BCCC',
    figures_dir=config.FIGURES_DIR / 'training' / 'bccc'
)

metrics_c = evaluate.evaluate_model(
    model_c, data_c,
    name='BCCC',
    dataset_tag='bccc',
    figures_dir=config.FIGURES_DIR / 'evaluation' / 'bccc'
)
print('BCCC metrics:', metrics_c)

## 13. Ablation study — BCCC-CIC-Bell-DNS-2024

In [ ]:
ablation_variants_c = {
    'FFN-Only': build_ffn_only(
        n_num_features=data_c['n_num_features'],
        dropout_rate=config.BCCC_DROPOUT_RATE,
        dropout_mid=config.BCCC_DROPOUT_MID,
        l2_reg=config.BCCC_L2_REG,
        lr=config.BCCC_LR,
        name='FFN_Only_BCCC'),
    'Encoder-Only': build_encoder_only(
        n_num_features=data_c['n_num_features'],
        vocab_size=data_c['vocab_size'],
        dropout_rate=config.BCCC_DROPOUT_RATE,
        dropout_mid=config.BCCC_DROPOUT_MID,
        l2_reg=config.BCCC_L2_REG,
        lr=config.BCCC_LR,
        name='Encoder_Only_BCCC'),
    'Full (paper)': model_c,
}

ablation_results_c = {}
for vname, vmodel in ablation_variants_c.items():
    if vname == 'Full (paper)':
        ablation_results_c[vname] = metrics_c
    else:
        print(f'\n--- Ablation variant: {vname} ---')
        train.train_model(vmodel, data_c,
                          name=f'ablation_bccc_{vname.lower().replace(" ", "_")}',
                          class_weight=class_weight_bccc)
        ablation_results_c[vname] = evaluate.evaluate_model(
            vmodel, data_c,
            name=f'BCCC — {vname}',
            dataset_tag=f'bccc_ablation_{vname.lower().replace(" ","_")}',
            figures_dir=config.FIGURES_DIR / 'ablation' / 'bccc'
        )

ablation_df_c = evaluate.ablation_comparison(
    ablation_results_c, 'BCCC',
    figures_dir=config.FIGURES_DIR / 'ablation' / 'bccc'
)
print(ablation_df_c)

### McNemar test — BCCC ablation

In [ ]:
y_true_c = data_c['y_te']
preds_c = {vn: _preds(vm, data_c) for vn, vm in ablation_variants_c.items()}

mcnemar_c = {}
for vname in ['FFN-Only', 'Encoder-Only']:
    result = evaluate.mcnemar_test(
        y_true_c,
        preds_c['Full (paper)'],
        preds_c[vname],
        name_a='Full (paper)',
        name_b=vname
    )
    mcnemar_c[f'Full vs {vname}'] = result

save_json(mcnemar_c, config.RESULTS_DIR / 'mcnemar_bccc.json')
print('McNemar tests saved.')

## 14. Per-category detection rate — BCCC

In [ ]:
if bccc_categories is not None:
    per_cat_df = evaluate.evaluate_per_category(
        model_c, data_c, bccc_categories,
        name='BCCC',
        figures_dir=config.FIGURES_DIR / 'evaluation' / 'bccc'
    )
    print(per_cat_df.to_string(index=False))
else:
    print('No category sidecar available.')

## 15. Training details for reproducibility

This cell reports the key training statistics required for reproducibility.

In [ ]:
def training_summary(history, name, data):
    h = history.history
    n_epochs   = len(h['loss'])
    best_epoch = int(np.argmin(h['val_loss'])) + 1
    summary = {
        'dataset': name,
        'total_epochs_run': n_epochs,
        'best_epoch': best_epoch,
        'best_val_loss': round(min(h['val_loss']), 4),
        'best_val_acc':  round(h['val_accuracy'][best_epoch-1], 4),
        'best_val_auc':  round(h['val_auc'][best_epoch-1], 4),
        'train_size':    int(len(data['y_tr'])),
        'val_size':      int(len(data['y_val'])),
        'test_size':     int(len(data['y_te'])),
        'n_num_features': data['n_num_features'],
        'vocab_size':    data['vocab_size'],
        'batch_size':    config.BATCH_SIZE,
        'max_seq_len':   config.MAX_SEQ_LEN,
        'tokenizer':     'Keras Tokenizer (from scratch on DNS data, OOV token: <OOV>)',
        'embedding_dim': config.EMBEDDING_DIM,
        'embedding_init': 'random uniform (trained end-to-end)',
    }
    return summary

ts_m = training_summary(history_m, 'Marques', data_m)
ts_c = training_summary(history_c, 'BCCC',    data_c)
save_json(ts_m, config.RESULTS_DIR / 'training_summary_marques.json')
save_json(ts_c, config.RESULTS_DIR / 'training_summary_bccc.json')

import pandas as pd
pd.DataFrame([ts_m, ts_c]).T

## 16. Final summary table

In [ ]:
summary = {
    'Marques (paper baseline)': {'Accuracy':0.9996,'Precision':0.9998,'Recall':0.9995,'F1-Score':0.9995,'ROC-AUC':None},
    'Marques (ours — Full)':    metrics_m,
    'Marques (ours — FFN-Only)':     ablation_results_m.get('FFN-Only', {}),
    'Marques (ours — Encoder-Only)': ablation_results_m.get('Encoder-Only', {}),
    'BCCC (ours — Full)':    metrics_c,
    'BCCC (ours — FFN-Only)':     ablation_results_c.get('FFN-Only', {}),
    'BCCC (ours — Encoder-Only)': ablation_results_c.get('Encoder-Only', {}),
}

summary_df = pd.DataFrame(summary).T
print(summary_df.to_string())
summary_df.to_csv(config.RESULTS_DIR / 'summary_all.csv')
print('\nSaved:', config.RESULTS_DIR / 'summary_all.csv')

evaluate.final_summary_plot(
    {k: v for k, v in summary.items() if v and all(v.values())},
    figures_dir=config.FIGURES_DIR
)

## 17. Save models and ZIP all figures for download

In [ ]:
import zipfile

# Save Keras models
model_m.save(config.MODELS_DIR / 'encoder_ffn_marques.keras')
model_c.save(config.MODELS_DIR / 'encoder_ffn_bccc.keras')

# Save scaler + tokenizer
for tag, data in [('marques', data_m), ('bccc', data_c)]:
    with open(config.MODELS_DIR / f'scaler_{tag}.pkl', 'wb') as f:
        pickle.dump(data['scaler'], f)
    with open(config.MODELS_DIR / f'tokenizer_{tag}.pkl', 'wb') as f:
        pickle.dump(data['tokenizer'], f)

print('Models and artefacts saved.')

# ZIP all figures (PDF + PNG)
zip_path = config.PROJECT_ROOT / 'figures_all.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in config.FIGURES_DIR.rglob('*'):
        if fpath.is_file() and fpath.suffix in ('.pdf', '.png'):
            zf.write(fpath, fpath.relative_to(config.PROJECT_ROOT))
print(f'Figures zipped → {zip_path}')

# On Colab, trigger download
if IS_COLAB:
    from google.colab import files
    files.download(str(zip_path))